# DS2002 · Saving Files on Colab (and Making Them Persist)

**Reference · Fall 2026**

---

Keep this notebook open in a tab. It is not a lecture and there is nothing to submit — it is the page you come back to when a file you wrote yesterday is gone today.

## The one thing to understand first

A Colab notebook runs on a temporary virtual machine that Google hands you and later takes back. That machine has a real filesystem at `/content`, and you can write to it all day. But when the runtime disconnects — you close the tab, you go idle for 90 minutes, you hit "Restart runtime", or you hit the ~12 hour session cap — **the machine is destroyed and every file on it goes with it.**

Your notebook code survives (it lives in your Google Drive as a `.ipynb`). Your *files* do not.

So there are exactly three places a file can live, and only two of them survive:

| Location | Path | Survives disconnect? |
|---|---|---|
| Colab runtime disk | `/content/...` | **No** |
| Your Google Drive | `/content/drive/MyDrive/...` | **Yes** |
| Your own laptop | wherever you downloaded it | **Yes** |

Everything below is about moving files between those three places on purpose instead of by accident.

## Am I even on Colab?

Every cell in this notebook is written to run both on Colab and on your laptop, so you can read it either place. The trick is one flag, set once, that the rest of the notebook checks.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('running on Colab:', IN_COLAB)
print('python:', sys.version.split()[0])
print('working directory:', Path.cwd())

On Colab that working directory prints `/content`. That is the default home for a notebook session, and it is the folder the file browser in the left sidebar shows you.

## Part 1 — Writing files to the runtime (fast, and temporary)

This is normal Python. There is nothing Colab-specific about it, which is exactly why people get burned: it works perfectly, right up until the runtime dies.

Use runtime disk for scratch work — intermediate files, a 2 GB download you are about to filter down, anything you can regenerate by re-running cells.

In [ ]:
# A working directory for scratch files. `parents=True, exist_ok=True` means
# this cell is safe to run twice, which is the whole point.
SCRATCH = Path('/content/scratch') if IN_COLAB else Path('./scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)

# Plain text, the long way — the `with` block closes the file for you.
note = SCRATCH / 'notes.txt'
with open(note, 'w') as f:
    f.write('Gate revenue looked wrong for the NC State game.\n')
    f.write('Suspect duplicate vendor rows.\n')

print('wrote:', note)
print('exists:', note.exists(), '| bytes:', note.stat().st_size)

In [ ]:
# Reading it back
print(note.read_text())

### Saving a DataFrame

`to_csv(index=False)` is the version you want almost every time. Without `index=False` pandas writes the row numbers as an unnamed first column, and then whoever reads the file next gets a mystery column called `Unnamed: 0`.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    'game': ['NC State', 'NC State', 'Louisville', 'Louisville'],
    'vendor': ['Hoos Burgers', 'Cav Merch North', 'Hoos Burgers', 'Rotunda Tacos'],
    'units': [120, 45, 98, 60],
    'revenue': [900.00, 1080.00, 735.00, 390.00],
})

csv_path = SCRATCH / 'orders.csv'
orders.to_csv(csv_path, index=False)
print('wrote', csv_path.stat().st_size, 'bytes to', csv_path)

# Read it straight back and confirm the shape survived the round trip.
back = pd.read_csv(csv_path)
print('rows out:', len(orders), '| rows in:', len(back))
back

### Other formats worth knowing

CSV is universal but lossy: every column comes back as text and gets re-guessed on read, so dates arrive as strings and integers can arrive as floats. When you are handing data to your own next notebook rather than to a human, use Parquet — it keeps the dtypes and is much smaller.

In [ ]:
# JSON — human-readable, good for nested records and for API-shaped data.
json_path = SCRATCH / 'orders.json'
orders.to_json(json_path, orient='records', indent=2)

# Parquet — keeps dtypes, small, not human-readable. Colab has the pyarrow
# engine preinstalled; a local Python may not, hence the guard.
pq_path = SCRATCH / 'orders.parquet'
try:
    orders.to_parquet(pq_path, index=False)
    has_parquet = True
except ImportError:
    has_parquet = False
    print('no parquet engine here — run `pip install pyarrow` (already present on Colab)')

for p in (csv_path, json_path) + ((pq_path,) if has_parquet else ()):
    print(f'{p.name:18} {p.stat().st_size:>7} bytes')

print()
print('dtypes after CSV round trip:')
print(pd.read_csv(csv_path).dtypes)

if has_parquet:
    print()
    print('dtypes after Parquet round trip:')
    print(pd.read_parquet(pq_path).dtypes)

In [ ]:
# See what is actually on the runtime disk right now.
for p in sorted(SCRATCH.iterdir()):
    print(f'{p.stat().st_size:>8} bytes  {p.name}')

## Part 2 — Getting files off Colab and onto your laptop

The quickest way to make a file permanent is to download it. Good for a finished deliverable, bad as a workflow you repeat twenty times.

In [ ]:
# Colab only. Fires a browser download; your browser may ask permission the first time.
if IN_COLAB:
    from google.colab import files
    files.download(str(csv_path))
else:
    print('Not on Colab — files.download() is unavailable. The file is already local:', csv_path)

Downloading many files one at a time is miserable. Zip the folder first and download one archive.

In [ ]:
import shutil

archive = shutil.make_archive(str(SCRATCH.parent / 'scratch_backup'), 'zip', root_dir=SCRATCH)
print('archive:', archive, '|', Path(archive).stat().st_size, 'bytes')

if IN_COLAB:
    from google.colab import files
    files.download(archive)

### Going the other direction: uploading from your laptop

`files.upload()` opens a file picker and drops whatever you choose into the current working directory. It returns a dict of `{filename: bytes}`.

Be aware of the tradeoff: an uploaded file lands on the *runtime* disk, so it disappears with the runtime and you get to pick it again next session. For anything you will use more than once, put it in Drive instead.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()          # opens the picker
    for name, data in uploaded.items():
        print(f'{name}: {len(data)} bytes')
    # Typical next step:
    # df = pd.read_csv(next(iter(uploaded)))
else:
    print('files.upload() is Colab-only.')

## Part 3 — Mounting Google Drive (the actual answer to persistence)

Mounting makes your Google Drive appear as a normal folder on the runtime's filesystem. After it succeeds, `open()`, `pd.read_csv()`, and `to_csv()` all work on Drive paths with no special syntax — you are just writing to a different directory.

Running the next cell pops up an authorization flow: pick your account, then approve access. You will do this once per runtime, every time you connect. It is a few clicks and it is unavoidable.

**One warning before you run it:** a mounted Drive is fully writable. `shutil.rmtree()` pointed at the wrong Drive path will really delete your files, and a notebook has no undo. Read your paths before you run destructive code.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')      # add force_remount=True if it acts stale
    print()
    print(os.listdir('/content/drive'))
else:
    print('Skipping mount — not on Colab.')

`/content/drive/MyDrive` is your own Drive. A few notes on paths:

- `MyDrive` has no space in it. The folder is displayed as "My Drive" in the web UI, and older tutorials use `/content/drive/My Drive` — that path still works but the spaces make shell commands painful. Prefer `MyDrive`.
- Shared drives, if you have them, appear under `/content/drive/Shareddrives/`.
- Files shared *with* you but not added to your Drive do **not** appear at all. Add them to your Drive first (Drive UI → right-click → "Add shortcut to Drive").

Now set up a course folder. Doing this once, in one place, is what keeps paths out of the rest of your code.

In [ ]:
if IN_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/DS2002/data')
else:
    DATA_DIR = Path('./data')          # local fallback so this notebook runs anywhere

DATA_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('writable:', os.access(DATA_DIR, os.W_OK))

That `DATA_DIR` pattern is worth internalizing. Define the folder once at the top of a notebook, build every path from it, and the same notebook runs on Colab and on your laptop with no edits. Hard-coded `/content/drive/MyDrive/...` strings scattered through thirty cells is how notebooks stop being portable.

In [ ]:
# Write to Drive. This is a persistent save — it is still there next week.
drive_csv = DATA_DIR / 'orders.csv'
orders.to_csv(drive_csv, index=False)

print('saved to:', drive_csv)
print('size:', drive_csv.stat().st_size, 'bytes')

In [ ]:
# Read it back the same way you would read any other file.
reloaded = pd.read_csv(drive_csv)
print(reloaded.shape)
reloaded.head()

In [ ]:
# What is in the folder?
for p in sorted(DATA_DIR.iterdir()):
    kind = 'dir ' if p.is_dir() else 'file'
    size = '' if p.is_dir() else f'{p.stat().st_size:>9} bytes'
    print(f'{kind} {size}  {p.name}')

### A timestamped save, so you never overwrite good output

`to_csv` overwrites silently. When you are producing something you might want to compare against later, put the timestamp in the filename.

In [ ]:
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
snapshot = DATA_DIR / f'orders_{stamp}.csv'
orders.to_csv(snapshot, index=False)
print('snapshot:', snapshot.name)

## Part 4 — The things you will actually save

Three cases that come up constantly in this course.

### Charts

`plt.savefig()` before `plt.show()`. In that order — `show()` clears the figure, so a `savefig()` after it writes a blank image. `bbox_inches='tight'` stops the axis labels getting cropped, and `dpi=150` is enough for a slide.

In [ ]:
import matplotlib.pyplot as plt

by_vendor = orders.groupby('vendor', as_index=False)['revenue'].sum()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(by_vendor['vendor'], by_vendor['revenue'])
ax.set_xlabel('Revenue ($)')
ax.set_title('Revenue by vendor')
fig.tight_layout()

chart_path = DATA_DIR / 'revenue_by_vendor.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')   # save first
plt.show()                                               # then display

print('chart saved:', chart_path, '|', chart_path.stat().st_size, 'bytes')

### A SQLite database

A `.db` file on Drive is a perfectly good persistent database for this course. Two rules: close the connection when you are done, and do not open the same Drive `.db` from two notebooks at once — SQLite's locking does not play well with Drive's syncing.

In [ ]:
import sqlite3

db_path = DATA_DIR / 'gameday.db'
conn = sqlite3.connect(db_path)

orders.to_sql('orders', conn, if_exists='replace', index=False)
conn.commit()

print(pd.read_sql_query(
    'SELECT vendor, SUM(revenue) AS revenue FROM orders GROUP BY vendor ORDER BY revenue DESC',
    conn,
))

conn.close()          # do not skip this
print()
print('db size:', db_path.stat().st_size, 'bytes')

### Copying a whole folder to Drive at the end of a session

If you did your work on the fast runtime disk (which is reasonable for big files), sweep the results into Drive before you disconnect.

In [ ]:
backup = DATA_DIR / 'scratch_backup'
shutil.copytree(SCRATCH, backup, dirs_exist_ok=True)   # dirs_exist_ok makes it re-runnable

print('copied to:', backup)
for p in sorted(backup.iterdir()):
    print('   ', p.name)

## Part 5 — Verifying the save actually happened

The failure that costs people a grade is believing a file is safe when it is not. Two habits prevent it.

**First: assert, do not assume.** One line after every save that matters.

In [ ]:
def confirm_saved(path, min_bytes=1):
    """Raise if `path` is missing or suspiciously empty."""
    path = Path(path)
    assert path.exists(), f'NOT SAVED: {path}'
    assert path.stat().st_size >= min_bytes, f'SAVED BUT EMPTY: {path}'
    print(f'OK  {path.stat().st_size:>9} bytes  {path}')
    return path


confirm_saved(drive_csv)
confirm_saved(chart_path)
confirm_saved(db_path)

**Second: check it in the Drive web UI.** Writes to a mounted Drive go through a sync layer, so a file can exist on the mount a few seconds before it appears at drive.google.com. If a file is large or you are about to disconnect, flush the mount and give it a moment.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()   # forces pending writes out
    print('unmounted — remount with drive.mount("/content/drive") to keep working')
else:
    print('Nothing to unmount.')

## Part 6 — Gotchas, in rough order of how often they bite

**"My file was there yesterday."** It was on `/content`, not Drive. There is no recovery. Save to `DATA_DIR` from now on.

**`FileNotFoundError` on a Drive path.** Either the runtime restarted and dropped the mount (re-run the mount cell — a restart clears it), or the parent folder does not exist. `open()` will not create missing folders; `Path.mkdir(parents=True, exist_ok=True)` will.

**Only the last part of the path exists.** `to_csv('/content/drive/MyDrive/DS2002/out/x.csv')` fails if `out/` is missing. Create the directory before writing to it, every time.

**Spaces in paths.** Fine in Python (`Path('My Drive')` works), a mess in shell magics where `!ls /content/drive/My Drive` splits into two arguments. Use `MyDrive`, or quote the path.

**A file you wrote is 0 bytes.** You used `open(...)` without a `with` block and never called `.close()`, so the buffer was never flushed. Always use `with`.

**Drive quota / "Input-output error".** Drive has a per-day write cap and, more commonly, a per-folder file count that gets slow. Writing 50,000 small files to Drive will fail or crawl. Write them to `/content` and copy over one zip instead.

**Restart vs. disconnect.** "Restart runtime" clears variables and unmounts Drive but keeps `/content` files. "Disconnect and delete runtime" destroys everything. Both require re-mounting.

**Secrets.** Never paste an API key into a notebook cell you will commit or share. Colab has a key icon in the left sidebar; store it there and read it with `userdata`:

```python
from google.colab import userdata
api_key = userdata.get('MY_API_KEY')
```

**Committing data to git.** Your notebook may live in a GitHub repo; your 200 MB CSV should not. Keep data in Drive and out of the repo, and keep `data/` in `.gitignore`.

## The short version

If you remember nothing else from this notebook:

```python
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/DS2002/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(DATA_DIR / 'output.csv', index=False)
assert (DATA_DIR / 'output.csv').exists()
```

Mount, define one folder, build every path from it, verify the write. Anything written outside `/content/drive/` is scratch — treat it as already deleted.

## Quick reference

| Task | Code |
|---|---|
| Detect Colab | `try: import google.colab; IN_COLAB=True` |
| Mount Drive | `from google.colab import drive; drive.mount('/content/drive')` |
| Force remount | `drive.mount('/content/drive', force_remount=True)` |
| Flush writes | `drive.flush_and_unmount()` |
| Make folders | `Path(p).mkdir(parents=True, exist_ok=True)` |
| Save DataFrame | `df.to_csv(path, index=False)` |
| Save with dtypes | `df.to_parquet(path, index=False)` |
| Save a chart | `fig.savefig(path, dpi=150, bbox_inches='tight')` before `plt.show()` |
| Save to SQLite | `df.to_sql('t', conn, if_exists='replace', index=False)` |
| Download to laptop | `from google.colab import files; files.download(str(path))` |
| Upload from laptop | `files.upload()` |
| Zip a folder | `shutil.make_archive('out', 'zip', root_dir=folder)` |
| Copy folder to Drive | `shutil.copytree(src, dst, dirs_exist_ok=True)` |
| Read a secret | `from google.colab import userdata; userdata.get('KEY')` |
| List a folder | `sorted(Path(p).iterdir())` |
| Free disk / RAM | `!df -h /content` and `!free -h` |